## Load Document

In [14]:
import os
import asyncio
from pathlib import Path
from dotenv import load_dotenv
from langchain_core.documents import Document
from typing import List
import numpy as np
from dotenv import load_dotenv
load_dotenv()

from pinecone import Pinecone, ServerlessSpec

In [ ]:
from llama_cloud import AsyncLlamaCloud

load_dotenv()

path = Path("../data/contracts")
pdf_paths = list(path.glob("*.pdf"))

async def main():
    if not pdf_paths:
        raise ValueError("No PDF files found.")

    client = AsyncLlamaCloud(api_key=os.getenv("LLAMA_CLOUD_API_KEY"))

    for pdf_path in pdf_paths:
        print(f"Processing {pdf_path.name}")

        with open(pdf_path, "rb") as f:
            file_obj = await client.files.create(file=f, purpose="parse")

        result = await client.parsing.parse(
            file_id=file_obj.id,
            tier="agentic_plus",
            version="latest",
            output_options={
                "markdown": {
                    "tables": {
                        "output_tables_as_markdown": True
                    }
                }
            },
            expand=["markdown"],
        )

        # Combine all pages into one clean LLM-ready string
        full_text = "\n\n".join(
            page.markdown for page in result.markdown.pages
        )

        # Save as .txt
        output_file = pdf_path.stem + "_parsed.txt"
        with open(output_file, "w", encoding="utf-8") as f:
            f.write(full_text)

        print(f"Saved: {output_file}")

await main()


Processing OnlineSvcsConsolidatedSLA(WW)(English)(January2026)CR (1).pdf
Saved: OnlineSvcsConsolidatedSLA(WW)(English)(January2026)CR (1)_parsed.txt
Processing SLA_69554920-7585-4a28-b2591717413708270_IDDBUYER1@3483.pdf
Saved: SLA_69554920-7585-4a28-b2591717413708270_IDDBUYER1@3483_parsed.txt


## Chunking

In [3]:
from langchain_community.document_loaders import TextLoader

processed_pdf = Path("../data/processed_contracts")
processed_pdf_path = list(processed_pdf.glob("*.txt"))
documents = []

for path in processed_pdf_path:
    loader = TextLoader(file_path=path, encoding='utf-8')
    docs = loader.load()
    documents.extend(docs)

print(documents)

e:\SentryChain-AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[Document(metadata={'source': '..\\data\\processed_contracts\\OnlineSvcsConsolidatedSLA(WW)(English)(January2026)CR (1)_parsed.txt'}, page_content='# Service Level Agreement for Microsoft Online Services\n## January 1, 2026\n\n# Table of Contents\n\n**TABLE OF CONTENTS ................................................................................................... 2**\n**INTRODUCTION ............................................................................................................ 4**\n**GENERAL TERMS .......................................................................................................... 5**\n**SERVICE SPECIFIC TERMS .............................................................................................. 7**\n**MICROSOFT DYNAMICS 365 .......................................................................................... 7**\n*   DYNAMICS 365 BUSINESS CENTRAL................................................................................ 7\n*   D

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size = 2000, chunk_overlap = 500):
    text_splitter = RecursiveCharacterTextSplitter(
        separators=["\nClause", "\nSection", " ", ""],
        chunk_overlap=chunk_overlap,
        chunk_size=chunk_size
    )

    split_docs = text_splitter.split_documents(documents=documents)
    print(f"Splitting {len(documents)} documents into {len(split_docs)} chunks")

    if split_docs:
        print(f"Content {split_docs[0].page_content[:100]}...")

    return split_docs

### Metadata:
* supplier
* section_id
* category(force majeure, penalty, termination)

In [5]:
def load_metadata(processed_pdf_path: list) -> list:
    docs = []

    for file_path in processed_pdf_path:
        with open(file_path, 'r', encoding = 'utf-8') as f:
            content = f.read()

        supplier_name = file_path.stem.split("-")[0]

        doc = Document(
            page_content=content,
            metadata = {
                "supplier": supplier_name,
                "doc_type": "SLA",
                "source": str(file_path)
            }
        )
        docs.append(doc)

    return docs

documents = load_metadata(processed_pdf_path=processed_pdf_path)

chunks = split_docs(documents=documents)
chunks

Splitting 2 documents into 328 chunks
Content # Service Level Agreement for Microsoft Online Services
## January 1, 2026

# Table of Contents

**T...


[Document(metadata={'supplier': 'OnlineSvcsConsolidatedSLA(WW)(English)(January2026)CR (1)_parsed', 'doc_type': 'SLA', 'source': '..\\data\\processed_contracts\\OnlineSvcsConsolidatedSLA(WW)(English)(January2026)CR (1)_parsed.txt'}, page_content='# Service Level Agreement for Microsoft Online Services\n## January 1, 2026\n\n# Table of Contents\n\n**TABLE OF CONTENTS ................................................................................................... 2**\n**INTRODUCTION ............................................................................................................ 4**\n**GENERAL TERMS .......................................................................................................... 5**\n**SERVICE SPECIFIC TERMS .............................................................................................. 7**\n**MICROSOFT DYNAMICS 365 .......................................................................................... 7**\n*   DYNAMICS 365 BUSINE

In [6]:
from langchain_community.graphs import Neo4jGraph
from langchain_experimental.graph_transformers import LLMGraphTransformer

In [7]:
from sentence_transformers import SentenceTransformer

texts = [chunk.page_content for chunk in chunks]

In [10]:
class EmbeddingManager:
    def __init__(self, model_name: str = "BAAI/bge-m3" ):
        self.model_name = model_name
        self.model = None
        self.load_model()

    def load_model(self):
        try:
            print(f"Loading model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model: {self.model_name}: {e}")

    def generate_embeddings(self, texts: List[str]):
        embeddings = self.model.encode(texts)
        return embeddings[0].astype(float).tolist()

embedding_manager = EmbeddingManager()

Loading model: BAAI/bge-m3
Model loaded successfully. Embedding dimension: 1024


In [12]:
pc = Pinecone(api_key=os.getenv('PINECONE_API_KEY'))
index_name= "sentry-chain-ai"
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=1024, 
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )
    print(f"Manual index '{index_name}' created.")

graph = Neo4jGraph(
    url=os.getenv("NEO4J_URI"), 
    username=os.getenv("NEO4J_USERNAME"), 
    password=os.getenv("NEO4J_PASSWORD")
)
for i, chunk in enumerate(chunks):
    const_id = f"{chunk.metadata['supplier']}#{i}"

    # push to neo4j
    graph.query("""
    MERGE (s:Supplier {name: $supplier})
    MERGE (c:Contract {name: 'SLA_2024'})
    MERGE (ch:Chunk {vector_id: $v_id})
    MERGE (s)-[:HAS_CONTRACT]->(c)
    MERGE (c)-[:HAS_DATA]->(ch)
    SET ch.text_preview = $text
    """, params={
        "supplier": chunk.metadata['supplier'],
        'v_id': const_id,
        "text": chunk.page_content 
    })

    # push to pinecone
    vector_values = embedding_manager.generate_embeddings([chunk.page_content])
    dense_index = pc.Index(index_name)
    dense_index.upsert(vectors=[{
        "id": const_id,
        "values": vector_values,
        "metadata": {**chunk.metadata, "text": chunk.page_content}
    }])

In [15]:
from langchain_groq import ChatGroq

llm = ChatGroq(api_key=os.getenv('GROQ_API_KEY'), model="llama-3.3-70b-versatile", temperature=0.1)